# Stride — train the floor-plan recognition model (Colab) — v5.2

Release tag on completion: `model-v5.1`. Same v5 generator, bigger training
budget: the first v5 run (10k samples / 30 epochs) learned the colored-CAD
dialect but paid a small accuracy tax on the older styles — the same budget
spread over ~30% more style diversity. 15k samples / 36 epochs gives the
extra coverage back. Expect roughly 7–9 hours on a T4; checkpoints save to
Drive every epoch, so a disconnect never loses the run.

**v5.2**: the train cell (step 6) now has a resume toggle — if Colab
disconnects mid-run, don't start over. Set `RESUME = True` and
`COMPLETED_EPOCHS` to whatever the last printed `epoch N ...` line showed,
then Run all again; dataset generation is deterministic (fixed seed) so it
reproduces the exact same data, and training continues from your saved
weights instead of from scratch.

Open this in Colab, set the runtime to a **GPU** (Runtime → Change runtime type → T4 GPU), then **Runtime → Run all**.

It clones the repo, generates the synthetic dataset, trains the U‑Net, exports ONNX, and saves `best.pt` + `stride-planseg.onnx` to your Google Drive (`MyDrive/stride-model/`).

Dataset generation restarts itself in fresh subprocesses every 400 samples (a leak in the native SVG renderer otherwise grows unbounded and can crash the process partway through a big run) and verifies the exact file count before continuing — a partial dataset now halts the run instead of silently training on it.


## 1. Mount Google Drive + check the GPU

The Drive permission popup appears **right here, at the start of the run** —
grant it once, and the rest of the run (including the 7–9h training) needs no
further interaction. Checkpoints and the final model save straight to
`MyDrive/stride-model/`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')   # popup appears NOW — grant access, then walk away

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('\n⚠️  No GPU. Runtime → Change runtime type → Hardware accelerator: T4 GPU, then Run all again.')


## 2. Config — tweak these, then Run all

In [ ]:
BRANCH  = 'main-uiyymm'   # branch to train from
SAMPLES = 15000           # training plans to generate (try 400 for a quick test)
VAL     = 500             # validation plans
EPOCHS  = 36              # training epochs (try 3 for a quick test)
BATCH   = 8               # lower to 4 if you hit out-of-memory
SIZE    = 512             # training crop size
BASE    = 32              # U-Net width (model capacity)


## 3. Clone the repo + install Node deps (for the generator)

In [ ]:
import os
if not os.path.isdir('stride'):
    !git clone --branch {BRANCH} https://github.com/tiienn/stride.git
%cd stride
!node --version
# only the generator's dep is needed (resvg); skip the app's heavy 3D deps
!npm install @resvg/resvg-js --no-save --no-audit --no-fund

## 4. Generate the synthetic dataset

Images + pixel‑perfect masks + ground‑truth JSON. ~110 ms/sample.

In [ ]:
import subprocess, glob

def run(cmd):
    print('$', ' '.join(cmd))
    subprocess.run(cmd, check=True)  # raises on non-zero exit -> halts Run All

# generate.mjs verifies its own output and exits non-zero if any file is
# missing (e.g. an OOM-killed chunk) - check=True turns that into a raised
# exception so a partial dataset can never be silently trained on.
run(['node', 'ml/generate.mjs', '--count', str(SAMPLES), '--out', 'ml/data/train', '--seed', '1'])
run(['node', 'ml/generate.mjs', '--count', str(VAL), '--out', 'ml/data/val', '--seed', '999'])

n_train = len(glob.glob('ml/data/train/img_*.png'))
n_val = len(glob.glob('ml/data/val/img_*.png'))
print('train images:', n_train)
print('val images:  ', n_val)
assert n_train == SAMPLES, f'train set incomplete: {n_train}/{SAMPLES} images - do not proceed'
assert n_val == VAL, f'val set incomplete: {n_val}/{VAL} images - do not proceed'

## 5. Preview a sample (sanity check)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].imshow(Image.open('ml/data/train/img_00000.png')); ax[0].set_title('drawing'); ax[0].axis('off')
ax[1].imshow(Image.open('ml/data/train/msk_00000.png')); ax[1].set_title('mask: red=wall green=door blue=window'); ax[1].axis('off')
plt.show()

## 6. Train

Drive was already mounted in step 1 (the mount call below is an instant
no-op safety). Checkpoints write straight to
`MyDrive/stride-model/checkpoints/` — a runtime reset mid-run no longer loses
the weights (last.pt is saved every epoch, best.pt whenever val mIoU improves).

**Resuming after a disconnect**: set `RESUME = True` below and
`COMPLETED_EPOCHS` to the epoch number from the last `epoch N  loss ...  val
IoU ...` line you saw printed before the disconnect (0-indexed — e.g. if
`epoch 27` was the last one printed, 28 epochs finished, so
`COMPLETED_EPOCHS = 28`). It continues from `last.pt` into a separate
`checkpoints/resume/` folder, so your original `checkpoints/best.pt` is
never at risk of being overwritten by a worse epoch — compare the two
printed mIoU numbers at the end and keep whichever is higher.

Watch the per-class `val IoU`. Walls climb first; doors/windows lag (rarer
pixels) but the class weights compensate.


In [ ]:
RESUME = False           # set True if resuming after a disconnect
COMPLETED_EPOCHS = 0     # epochs already finished (see markdown above)

from google.colab import drive
drive.mount('/content/drive')
import os
CKPT = '/content/drive/MyDrive/stride-model/checkpoints'
os.makedirs(CKPT, exist_ok=True)
%cd /content/stride/ml/train

if not RESUME:
    !python train.py --data ../data/train --out {CKPT} \
        --epochs {EPOCHS} --batch {BATCH} --size {SIZE} --base {BASE} --val-frac 0.05
else:
    # generate.mjs above used a fixed --seed, and train.py's val split uses a
    # fixed manual_seed(0), so the regenerated dataset and split are byte-for-
    # byte identical to the original run - resuming trains on the same data.
    # Writes to a SEPARATE folder so the original best.pt/last.pt (already on
    # Drive, your safety net) are never overwritten. --lr picks up near where
    # the original 36-epoch cosine schedule would have been at
    # COMPLETED_EPOCHS, then anneals to ~0 over the remaining epochs.
    import math
    remaining = EPOCHS - COMPLETED_EPOCHS
    assert remaining > 0, 'COMPLETED_EPOCHS must be less than EPOCHS'
    resume_lr = 3e-4 * (1 + math.cos(math.pi * COMPLETED_EPOCHS / EPOCHS)) / 2
    RESUME_CKPT = CKPT + '/resume'
    os.makedirs(RESUME_CKPT, exist_ok=True)
    print(f'resuming from epoch {COMPLETED_EPOCHS}: {remaining} epochs left, lr {resume_lr:.2e} -> ~0')
    !python train.py --data ../data/train --out {RESUME_CKPT} --resume {CKPT}/last.pt \
        --epochs {remaining} --lr {resume_lr} --batch {BATCH} --size {SIZE} --base {BASE} --val-frac 0.05
    print(f'\nDone. Compare the final mIoU above against your pre-disconnect run.')
    print(f'If {RESUME_CKPT}/best.pt scored higher, copy it over {CKPT}/best.pt')
    print(f'before running the export cell below (step 8).')
%cd /content/stride


## 7. Try it on a held‑out plan

In [ ]:
%cd /content/stride/ml/train
!python infer.py --checkpoint {CKPT}/best.pt --base {BASE} \
    --image ../data/val/img_00003.png --out-prefix /content/pred
%cd /content/stride
from PIL import Image
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].imshow(Image.open('ml/data/val/img_00003.png')); ax[0].set_title('held-out drawing'); ax[0].axis('off')
ax[1].imshow(Image.open('/content/pred_mask.png')); ax[1].set_title('model prediction'); ax[1].axis('off')
plt.show()

## 8. Export ONNX (for in‑browser inference in Stride)

In [ ]:
!pip install --quiet onnx
%cd /content/stride/ml/train
import torch
from model import UNet
m = UNet(4, base=BASE)
m.load_state_dict(torch.load(f'{CKPT}/best.pt', map_location='cpu'))
m.eval()
# dynamo=False forces the legacy TorchScript-based exporter: one
# self-contained .onnx file with weights embedded. The new dynamo/onnxscript
# exporter (Colab's default since PyTorch 2.9) splits weights into a
# separate .onnx.data file even for a 30MB model - onnxruntime-web and our
# single-URL fetch-model.mjs both expect one file, so opt out of that.
torch.onnx.export(m, torch.zeros(1, 3, 512, 512), '/content/drive/MyDrive/stride-model/stride-planseg.onnx',
                  input_names=['image'], output_names=['logits'],
                  dynamic_axes={'image': {2: 'h', 3: 'w'}, 'logits': {2: 'h', 3: 'w'}},
                  opset_version=17, dynamo=False)
print('exported -> MyDrive/stride-model/stride-planseg.onnx (single file)')
%cd /content/stride


## 9. Verify what's on Google Drive

Everything already saved during training/export — this just lists it. Download
`best.pt` and `stride-planseg.onnx` from Drive and publish them as a GitHub
release (tag `model-v5.1`).


In [ ]:
!ls -la /content/drive/MyDrive/stride-model/ /content/drive/MyDrive/stride-model/checkpoints/
!ls -la /content/drive/MyDrive/stride-model/checkpoints/resume/ 2>/dev/null || true
print('\nDone: download best.pt + stride-planseg.onnx from Drive -> GitHub release model-v5.1')
